# Larger systems with the stochastic network model

A morphology from molecular dynamics holds a few thousand molecules. For charge transport in a disordered material this is often far too small: a carrier in equilibrium sits in the low-energy tail of the density of states (DOS), and a small box samples that tail badly. The mobility of a small box then depends on its few deepest sites and changes by orders of magnitude from one box to the next.

The stochastic network model (Baumeier, Stenzel, Poelking, Andrienko, Schmidt, Phys. Rev. B 86, 184202 (2012)) is a random hopping network with the statistics of the morphology that can be generated at any size. This tutorial

1. fits the model to two morphologies of 1000 molecules with `networkfit`: MADN (moderate disorder) and bathocuproine, BCP (strong disorder and a trap),
2. checks the fit against the morphology, including the transport,
3. generates larger networks with `networkgen` and follows the mobility with system size,
4. looks at the trap in BCP and how much the result depends on the trap density,
5. shows why averaging over many small boxes does not replace a large one.

The theory is in the XTP manual, section *Finite systems and the stochastic network model*; the options in *Transport Simulations*.

## Requirements
* VOTCA installed and the environment activated with `VOTCARC.bash`.
* Python with `numpy` and `matplotlib`.

The calculations take about 15 minutes in total. In Jupyter, `!` runs a shell command.

In [ ]:
import glob
import gzip
import os
import shutil
import subprocess
import numpy as np
import matplotlib.pyplot as plt

kT = 8.617333e-5 * 300  # eV


def run(command):
    # run a VOTCA command, return its output (raises if it fails)
    p = subprocess.run(command, shell=True, check=True, capture_output=True, text=True)
    return p.stdout


def results(filename):
    # read a 'key value unit' result file into a dictionary
    values = {}
    with open(filename) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            key, value = line.split()[:2]
            try:
                values[key] = float(value)
            except ValueError:
                values[key] = value
    return values


def show_log(log, first="Read "):
    # print the log of a tool from the line starting with `first`, without the banner
    lines = log.splitlines()
    start = next(i for i, l in enumerate(lines) if first in l)
    print("\n".join(l.replace("... ... ", "") for l in lines[start:] if l.strip() and l.strip() != "...."))


def mobility(statefile, carrier="hole", field=1e7):
    # exact mobility (cm^2/Vs) of one carrier in a periodic box (kmcmultiple, master equation)
    tag = os.path.splitext(statefile)[0] + "_" + carrier
    run(f'xtp_run -e kmcmultiple -c carriertype={carrier} "field={field} 0 0" runtime=1000 outputtime=0 '
        f'ratefile={tag}_rates.dat occfile={tag}_occ.dat resultfile={tag}.res -f {statefile}')
    for f in (f"{tag}_rates.dat", f"{tag}_occ.dat"):
        os.remove(f)
    if os.path.basename(statefile).startswith("model_"):
        os.remove(statefile)  # model networks are large; keep the disk clean
    return results(f"{tag}.res")["exact_mobility"]


def generate(parameters, name, sites=None, seed=1, trapfactor=1.0, energyfile=False):
    # write a model network with networkgen; returns the state file name
    if os.path.exists(name + ".hdf5"):
        os.remove(name + ".hdf5")
    options = f"job_name={name} parameters={parameters} seed={seed} trapfactor={trapfactor}"
    if sites is not None:
        options += f" sites={sites}"
    if energyfile:
        options += f" energyfile={name}_E.txt"
    run(f"xtp_tools -e networkgen -c {options}")
    return name + ".hdf5"


def stats_blocks(filename):
    # the blocks of a networkfit statsfile: {title: array}
    blocks = {}
    for block in open(filename).read().split("\n\n\n"):
        lines = [l for l in block.strip().splitlines()]
        if len(lines) < 3:
            continue
        title = lines[0][2:]
        blocks[title] = np.array([[float(x) for x in l.split()] for l in lines[2:]])
    return blocks


# the morphologies are shipped compressed
for name in ("MADN", "BCP"):
    with gzip.open(f"{name}.hdf5.gz", "rb") as fin, open(f"{name}.hdf5", "wb") as fout:
        shutil.copyfileobj(fin, fout)
# networkgen does not overwrite state files: remove those of a previous run
for f in glob.glob("model_*.hdf5"):
    os.remove(f)

## The morphologies

Both state files contain 1000 molecules in a cubic box (MADN 8.8 nm, BCP 8.1 nm) with everything the transport calculators need: centers of mass, site energies (electrostatic and polarization contributions from polarizable force fields, plus internal energies), electronic couplings $|J|^2$ for holes and electrons in a neighbor list, and reorganization energies. For MADN only the holes have site energies. To keep the files small, each molecule is represented by one pseudo atom at its center of mass.

## Fitting the model to MADN

`networkfit` determines all parameters of the model:

* the **point process** of the sites, from the nearest-neighbor distance distribution $D(r)$, the spherical contact distribution $H(r)$ and the pair correlation function $g(r)$,
* the **site energies**: mean, width $\sigma$ and the spatial correlation $\kappa(r)$, described by a moving average over the $l_b$ and $l_c$ nearest sites,
* the **connection probability** $f(r)$ of two sites at distance $r$,
* the **couplings**: mean and variance of $\log_{10}|J|^2$ against $r$ and the distribution of the residuals,
* **traps** from the slow modes of the master equation,
* for two carriers, the correlation of their site energies and couplings.

It then generates `validate` (4) model boxes of the same size and compares their statistics with the morphology. The fit of the point process takes about two minutes.

In [ ]:
log = run("xtp_tools -e networkfit -c job_name=MADN statefile=MADN.hdf5 'carriers=h e' "
          "statsfile=MADN_stats.dat energyfile=MADN_E.txt")
show_log(log)

The model has no traps for MADN. The width of the hole DOS is about $3.7\,k_BT$ (after correcting the sample value for the finite box, see the manual). The validation table compares the morphology with the model boxes: geometry, energies, couplings, the energy correlation at a few distances and the exact zero-field diffusion coefficient of one hole. The morphology lies within the spread of the model boxes.

The parameter file is plain XML and can be edited:

In [ ]:
print(open("MADN_network.xml").read())

The fitted functions from the `statsfile`:

In [ ]:
b = stats_blocks("MADN_stats.dat")
fig, ax = plt.subplots(2, 3, figsize=(14, 8))
for a, key, ylabel in ((ax[0, 0], "nearest neighbour distance distribution D(r)", "D(r)"),
                       (ax[0, 1], "spherical contact distribution H(r)", "H(r)"),
                       (ax[0, 2], "pair correlation function g(r)", "g(r)")):
    d = b[key]
    a.plot(d[:, 0], d[:, 1], "k-", label="morphology")
    a.plot(d[:, 0], d[:, 2], "r--", label="model (mean of boxes)")
    a.set_xlabel("r (nm)"); a.set_ylabel(ylabel); a.legend()
d = b["pairs per site in distance bins"]
ax[1, 0].plot(d[:, 0], d[:, 1], "k-", label="morphology")
ax[1, 0].plot(d[:, 0], d[:, 2], "r--", label="model")
f = b["connection probability f(r)"]
ax[1, 0].plot(f[:, 0], f[:, 1], "b:", label="f(r)")
ax[1, 0].set_xlabel("r (nm)"); ax[1, 0].set_ylabel("pairs per site / f(r)"); ax[1, 0].legend()
d = b["energy correlation kappa(r), carrier h"]
ax[1, 1].plot(d[:, 0], d[:, 1], "k.", ms=3, label="morphology")
ax[1, 1].plot(d[:, 0], d[:, 2], "r-", label="moving average")
ax[1, 1].set_xlabel("r (nm)"); ax[1, 1].set_ylabel(r"$\kappa(r)$"); ax[1, 1].set_ylim(-0.2, 1.0); ax[1, 1].legend()
d = b["mean log10(J^2/eV^2), carrier h"]
ax[1, 2].plot(d[:, 0], d[:, 1], "k-", label="morphology")
ax[1, 2].plot(d[:, 0], d[:, 2], "r--", label="model")
ax[1, 2].set_xlabel("r (nm)"); ax[1, 2].set_ylabel(r"mean $\log_{10}(|J|^2/\mathrm{eV}^2)$"); ax[1, 2].legend()
plt.tight_layout()
plt.show()

The correlation of the site energies reaches far: the best long-range average uses $l_c\approx 300$ sites, a third of the box. How well this range is determined shows the quality of the fit for every $l_c$:

In [ ]:
scan = b["rms of the best kappa fit for each l_c, carrier h"]
plt.plot(scan[:, 0], scan[:, 1], "o-")
plt.xlabel("$l_c$")
plt.ylabel(r"rms deviation of $\kappa(r)$")
plt.show()

The minimum is shallow: correlations over distances comparable to the box are not well determined by a box of 1000 molecules.

## Transport: morphology, model boxes, larger networks

We compare the exact mobility of one hole at $10^7$ V/m (from the master equation, computed by `kmcmultiple`; no sampling noise) in the morphology with model boxes of the same size, and then with larger networks. Each model box is a new realization (`seed`).

In [ ]:
mu_madn = mobility("MADN.hdf5")
print(f"MADN morphology: {mu_madn:.3e} cm^2/Vs")

madn = {}
for n, seeds in ((1000, range(1, 13)), (8000, range(1, 4))):
    madn[n] = [mobility(generate("MADN_network.xml", f"model_madn_{n}_{s}", sites=n, seed=s)) for s in seeds]
    print(n, "sites:", " ".join(f"{m:.2e}" for m in madn[n]))

In [ ]:
for n, values in madn.items():
    plt.semilogy([n] * len(values), values, "o", alpha=0.6)
    plt.semilogy(n, np.exp(np.mean(np.log(values))), "k_", ms=25, mew=2)
plt.axhline(mu_madn, color="r", label="morphology (1000 molecules)")
plt.xscale("log")
plt.xlabel("sites")
plt.ylabel(r"$\mu$ (cm$^2$/Vs)")
plt.legend()
plt.title("MADN holes: model networks (black: geometric mean)")
plt.show()

The morphology lies within the spread of the model boxes of the same size: the model reproduces its transport. The boxes of 1000 sites scatter by a factor of a few, single boxes by more than an order of magnitude; boxes of 8000 sites scatter less, and their mean is lower. Even 8000 sites are not converged: for $\sigma\approx3.7\,k_BT$ the criterion of Baumeier et al., $(\sigma/k_BT)^2\le-5.7+1.05\ln N$, asks for about $10^8$ sites. In a study one would continue to $10^5$–$10^6$ sites (the exact solution takes minutes for $10^5$ sites; beyond that, use the kinetic Monte Carlo results) and extrapolate.

## BCP: strong disorder and a trap

BCP has a much broader DOS. The fit also looks for traps:

In [ ]:
log = run("xtp_tools -e networkfit -c job_name=BCP statefile=BCP.hdf5 'carriers=h e' "
          "statsfile=BCP_stats.dat energyfile=BCP_E.txt")
show_log(log)

For holes `networkfit` finds a single-molecule trap: one molecule 0.9 eV below the mean holds practically all of the equilibrium weight. A hole leaves it at a rate of about $10^{-3}$ s$^{-1}$, although it hops out at about $10^3$ s$^{-1}$: almost every hop out is followed by a hop back. For electrons there is a trap, too. The trap is far outside the Gaussian DOS of the rest of the morphology:

In [ ]:
E = np.loadtxt("BCP_E.txt")  # id, x, y, z, E_h, E_e, hole trap, electron trap
eh, trap = E[:, 4], E[:, 6]
bulk = eh[trap == 0]
x = np.linspace(eh.min() - 0.1, eh.max() + 0.1, 300)
plt.hist(eh, bins=50, density=True, alpha=0.5, label="morphology")
plt.plot(x, np.exp(-(x - bulk.mean())**2 / (2 * bulk.std()**2)) / np.sqrt(2 * np.pi) / bulk.std(), "k-",
         label=f"Gaussian, $\\sigma$ = {bulk.std():.3f} eV = {bulk.std() / kT:.1f} $k_BT$")
for e in eh[trap == 1]:
    plt.axvline(e, color="r", ls="--", label="hole trap")
plt.axvline(bulk.mean() - bulk.std()**2 / kT, color="g", ls=":", label=r"equilibrium energy $-\sigma^2/k_BT$")
plt.xlabel("hole site energy (eV)")
plt.ylabel("density")
plt.legend()
plt.show()
s = bulk.std() / kT
print(f"sigma/kT = {s:.2f}; sites needed by the criterion: {np.exp((s**2 + 5.7) / 1.05):.1e}")

In an infinite Gaussian DOS of this width a hole would sit around $-\sigma^2/k_BT$, far below every site of the box. With $\sigma/k_BT\approx7.5$ the criterion asks for about $10^{25}$ sites, which no simulation reaches.

The trap is stored as a template in the parameter file (its energies for both carriers relative to the bulk mean) with the density of the morphology, one per 1000 sites; `networkgen` inserts it at random places. How much does it matter?

In [ ]:
mu_bcp = mobility("BCP.hdf5")
print(f"BCP morphology: {mu_bcp:.3e} cm^2/Vs")
bcp_traps = [mobility(generate("BCP_network.xml", f"model_bcp_t{s}", seed=s)) for s in range(1, 9)]
bcp_plain = [mobility(generate("BCP_network.xml", f"model_bcp_p{s}", seed=s, trapfactor=0)) for s in range(1, 9)]
print("without traps:", " ".join(f"{m:.1e}" for m in bcp_plain))
print("with traps:   ", " ".join(f"{m:.1e}" for m in bcp_traps))
plt.semilogy([1] * 8, bcp_plain, "o", label="model, Gaussian only")
plt.semilogy([2] * 8, bcp_traps, "s", label="model with the trap template")
plt.axhline(mu_bcp, color="r", label="morphology")
plt.xticks([1, 2], ["no traps", "with traps"])
plt.xlim(0.5, 2.5)
plt.ylabel(r"$\mu$ (cm$^2$/Vs)")
plt.title("BCP holes, boxes of 1000 sites")
plt.legend()
plt.show()

Without the trap the model boxes are four to eight orders of magnitude faster than the morphology; with one trap per box most come within a factor of a few (the trap template is the same, its surroundings differ). A Gaussian model alone cannot describe BCP.

The trap density, however, is the least certain parameter: one trap in 1000 molecules fixes it only to between 0.025 and 5.6 times $10^{-3}$ per site (95 % Poisson interval). `trapfactor` multiplies the density; a scan with networks of 8000 sites:

In [ ]:
factors = [0, 0.25, 1, 4]
scan = {f: [mobility(generate("BCP_network.xml", f"model_bcp8k_{f}_{s}", sites=8000, seed=s, trapfactor=f))
            for s in (1, 2)] for f in factors}
for i, f in enumerate(factors):
    plt.semilogy([i] * 2, scan[f], "o")
plt.axhline(mu_bcp, color="r", label="morphology (1000 molecules)")
plt.xticks(range(len(factors)), [f"{f}\n({int(f * 8)} traps)" for f in factors])
plt.xlabel("trapfactor (hole traps in 8000 sites)")
plt.ylabel(r"$\mu$ (cm$^2$/Vs)")
plt.legend()
plt.show()

The mobility is set by the trap density over many orders of magnitude. A prediction for BCP therefore needs the trap density from a larger or independent source (more morphologies, or the chemistry of the trap); the model makes this dependence explicit instead of hiding it in one box.

## Many small boxes are not one large box

With such broad distributions it is tempting to average over many small boxes. We generate 24 boxes of 1000 sites and 4 of 8000 sites of the Gaussian BCP model (no traps; two of the large ones are reused from the scan above) and compare averages of groups of 8 small boxes with the large boxes:

In [ ]:
small, lnZ = [], []
for s in range(1, 25):
    state = generate("BCP_network.xml", f"model_bcp_small_{s}", seed=100 + s, trapfactor=0, energyfile=True)
    small.append(mobility(state))
    x = -np.loadtxt(f"model_bcp_small_{s}_E.txt")[:, 4] / kT      # hole energies
    lnZ.append(x.max() + np.log(np.exp(x - x.max()).sum()))         # ln of the Boltzmann sum
small, lnZ = np.array(small), np.array(lnZ)
large = scan[0] + [mobility(generate("BCP_network.xml", f"model_bcp8k_0_{s}", sites=8000, seed=s, trapfactor=0))
                   for s in (3, 4)]

rng = np.random.default_rng(1)
means = {"arithmetic": [], "geometric": [], "harmonic": [], "Boltzmann-weighted": []}
for _ in range(5000):
    k = rng.choice(len(small), 8, replace=False)
    w = np.exp(lnZ[k] - lnZ[k].max())
    means["arithmetic"].append(np.log10(small[k].mean()))
    means["geometric"].append(np.log10(small[k]).mean())
    means["harmonic"].append(np.log10(8 / np.sum(1 / small[k])))
    means["Boltzmann-weighted"].append(np.log10((w * small[k]).sum() / w.sum()))
print("log10 mobility (cm^2/Vs), mean +- standard deviation")
print(f"  single boxes of 1000 sites   {np.log10(small).mean():6.2f} +- {np.log10(small).std():.2f}")
print(f"  boxes of 8000 sites          {np.log10(large).mean():6.2f} +- {np.log10(large).std():.2f}")
for key, v in means.items():
    print(f"  {key:19s} of 8 small {np.mean(v):6.2f} +- {np.std(v):.2f}")

The mobilities of the small boxes spread over orders of magnitude, and their plain means do not agree with the large boxes: the arithmetic mean is dominated by rare fast boxes, the harmonic mean by rare slow ones, and the geometric mean describes a typical box, which is not what a large system does.

The reason is the equilibrium of a single carrier. In a large system made of sub-boxes $k$ the carrier is found in sub-box $k$ with the probability $P_k = Z_k/\sum_l Z_l$, with the Boltzmann sum $Z_k=\sum_{A\in k}e^{-E_A/k_BT}$, and the drift velocity is $\mathbf v=\sum_k P_k\mathbf v_k$. If the sub-boxes behave as isolated boxes, the mobility of the large system is the Boltzmann-weighted mean $\sum_k Z_k\mu_k/\sum_k Z_k$, which agrees with the boxes of 8000 sites much better. It is still an approximation: carriers also leave a deep region across the boundaries of a sub-box, which isolated boxes do not have.

For a Gaussian DOS, $Z/N$ fluctuates from box to box unless $N\gg e^{(\sigma/k_BT)^2}$. Below that, $Z$ is dominated by the deepest site of the box, and so is the mobility. Many realizations of small boxes are therefore, at best, one larger box in pieces: they do not converge to the mobility of a large system. Converged values need large enough networks, or an extrapolation in size or temperature (Kordt et al., J. Chem. Theory Comput. 10, 2508 (2014)).

## Summary

* `networkfit` turns a morphology into a stochastic network model; check the validation table, the fitted functions and the transport of model boxes of the same size.
* `networkgen` generates networks of any size; follow the mobility with size until it stops changing, using several realizations per size.
* For broad DOS, finite boxes are dominated by their deepest sites; plain averages over many small boxes are biased.
* Traps found in the morphology are inserted as templates; their density is uncertain and should be scanned with `trapfactor`.